# Benchmark on grouping v2 folds: RT-DETR-L (3 seeds), two parts

Completes the RT-DETR-L row on the v2 folds (docs/BENCHMARK_V2_PLAN.md). Settings identical to the v1 RT-DETR runs (notebook 09): 100 epochs, no early stopping, `last.pt` (EMA), batch 16, imgsz 640, AMP on, Ultralytics 8.4.215 defaults.
About 95-100 min per run on T4 with two runs in parallel, so the 15 runs are split in two parts of about 6.5 h each (Kaggle sessions stop at 12 h):
* **PART = "A"**: seed 42 folds 0-4 + seed 43 folds 0-2 = 8 runs
* **PART = "B"**: seed 43 folds 3-4 + seed 44 folds 0-4 = 7 runs

Run A and B as separate Save Versions (two accounts, or one after the other). `--save-val-preds` adds the shared-evaluator parity check (column "parity |diff|", must be about 0).
Pre-declared failure rule (same as v1): a run with test mAP50-95 below 0.01 or NaN validation is a failure, kept in the "as measured" mean, never re-run or replaced.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
# faster-coco-eval is pre-installed so Ultralytics' save_json step never auto-installs it in the middle of a run
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215", "faster-coco-eval>=1.6.7"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert "--save-val-preds" in open(f"{REPO}/scripts/training/run_baseline.py").read(), "repo is behind: push the --save-val-preds patch first"
from ultralytics import YOLO, RTDETR

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = False
PART = "A"                     # "A" or "B"
MODEL = "rtdetr-l.pt"
RTDETR(MODEL); YOLO("yolo11n.pt")       # download once here so the two GPU workers do not race
V2 = f"{REPO}/data_splits_v2"
assert os.path.isdir(f"{V2}/clean5")
RUNS = {"A": [(42, f) for f in range(5)] + [(43, f) for f in (0, 1, 2)],
        "B": [(43, f) for f in (3, 4)] + [(44, f) for f in range(5)]}[PART]
if SMOKE:
    EPOCHS, OUT, ZIP = 2, f"/kaggle/working/results/v2r_{PART}_smoke", f"v2r_{PART}_smoke_results"; RUNS = RUNS[:1]
else:
    EPOCHS, OUT, ZIP = 100, f"/kaggle/working/results/v2r_{PART}", f"v2r_{PART}_results"

def make_cmd(fold, seed):
    def f(gpu):
        return [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", "clean", "--fold", str(fold), "--seed", str(seed),
                "--model", MODEL, "--epochs", str(EPOCHS), "--batch", "16", "--fixed-epochs", "--device", str(gpu),
                "--data-root", DATA_ROOT, "--splits-dir", V2, "--out", OUT, "--save-val-preds"]
    return f
TASKS = [(f"v2rclean_f{f}_s{s}_rtdetr-l", make_cmd(f, s)) for s, f in RUNS]
print(len(TASKS), "runs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn = q.get_nowait()
        except queue.Empty:
            return
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import glob, json, shutil
print(f"{'run':40s} {'mAP50-95':>9s} {'mAP50':>7s} {'train_min':>9s} {'parity |diff|':>13s}  flag")
for p in sorted(glob.glob(f"{OUT}/*/metrics.json")):
    r = json.load(open(p)); m = r["test"]["map5095"]; par = r.get("test_offline_parity", {})
    d = par.get("abs_diff_map5095"); flag = "<-- collapsed? check the log" if m < 0.05 else ""
    if d is None:
        flag += " parity: " + str(par.get("error", "missing"))
    elif d > 1e-3:
        flag += " <-- parity off, tell Claude"
    print(f"{r['tag']:40s} {m:9.4f} {r['test']['map50']:7.4f} {r['train_seconds']/60:9.1f} {('%.2e' % d) if d is not None else 'n/a':>13s}  {flag}")
shutil.make_archive(f"/kaggle/working/{ZIP}", "zip", OUT)
print("\nzip:", f"/kaggle/working/{ZIP}.zip"); print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `v2r_<PART>_results.zip`, unzip once into `results/` (any folder name; Claude renames it to `results/v2r_<PART>`), then paste the printed table to Claude.